# Nordea Case Study: IBM HR Analytics Employee Attrition
This notebook focuses on building and evaluating predictive models for employee attrition. It covers 
- data preprocessing, 
- model training, 
- validation, and 
- performance comparison.



## 1. Setup and Data Loading

In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

import joblib

# Preprocessing & pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Model selection
from sklearn.model_selection import train_test_split, cross_val_score

# Metrics
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    confusion_matrix,
    f1_score
)

In [13]:

# Set style
sns.set_theme(style="whitegrid")

# Load the dataset
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Display first few rows
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [14]:
df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

## 2. Data Preprocessing & Feature Engineering

Prepare the dataset for modeling by:

- Removing constant (non-informative) features  
- Converting target and key variables to numeric format  
- Encoding categorical variables using one-hot encoding  
- Creating new features to capture business-relevant patterns, such as:
  - Compensation relative to job level  
  - Career progression speed  
  - Employee stability  
  - Work intensity and travel burden  
- Dropping redundant features to reduce multicollinearity  

Note: Feature scaling (standardization) is not applied at this stage, as it is handled within model-specific pipelines during training.

In [15]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns:", constant_cols)
df = df.drop(columns=constant_cols)


Constant columns: ['EmployeeCount', 'Over18', 'StandardHours']


In [16]:
# convert to numeric data
df['Attrition'] = df['Attrition'].map({'Yes':1, 'No':0})
df.OverTime = df.OverTime.map({'No':0,'Yes':1})

# hot encode the rest
df = pd.get_dummies(df, drop_first=True, dtype=int)

df['Income_per_Level'] = df['MonthlyIncome'] / (df['JobLevel'] + 1)
df['Years_per_Promotion'] = df['YearsAtCompany'] / (df['YearsSinceLastPromotion'] + 1)
df['Company_Stability'] = df['YearsAtCompany'] / (df['NumCompaniesWorked'] + 1)
df['Work_Intensity'] = df['OverTime'] * (5 - df['WorkLifeBalance'])
df['Travel_Burden'] = df['BusinessTravel_Travel_Frequently'] * df['DistanceFromHome']

df = df.drop(columns=['MonthlyIncome'])

## 3. Train-Test Split

Split the dataset into training and test sets to evaluate model performance on unseen data. Stratification is applied to preserve the class distribution of the target variable.

In [17]:


# Separate features and target
X = df.drop(columns=['Attrition'])
y = df['Attrition']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important for imbalance
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1176, 49)
Test shape: (294, 49)


## 4. Model Training and Selection

Train and evaluate multiple models to identify the best-performing approach for predicting employee attrition.

- Compare different algorithms (Logistic Regression, Random Forest, CatBoost)  
- Evaluate the impact of class imbalance handling using SMOTE  
- Use cross-validation (ROC-AUC) to assess model stability  
- Evaluate performance on a held-out test set  
- Adjust the classification threshold to prioritize recall for identifying employees at risk of leaving  

Given the business context, recall is prioritized to minimize missed at-risk employees, even at the cost of lower precision.

Note: Ensembling techniques are not considered, as the focus is on maintaining model interpretability and ease of explanation.

In [24]:
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

results = []

for name, model in models.items():
    for use_smote in [True, False]:

        # --- Pipeline ---
        if name == "Logistic":
            if use_smote:
                pipeline = ImbPipeline([
                    ('scaler', StandardScaler()),
                    ('smote', SMOTE(random_state=42)),
                    ('model', model)
                ])
            else:
                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('model', model)
                ])
        else:
            if use_smote:
                pipeline = ImbPipeline([
                    ('smote', SMOTE(random_state=42)),
                    ('model', model)
                ])
            else:
                pipeline = model

        # --- CV ---
        cv_score = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=5,
            scoring='roc_auc'
        ).mean()

        # --- Fit ---
        pipeline.fit(X_train, y_train)

        # --- Predict probabilities ---
        y_proba = pipeline.predict_proba(X_test)[:,1]
        roc = roc_auc_score(y_test, y_proba)

        # --- Threshold loop ---
        for t in thresholds:

            y_pred = (y_proba > t).astype(int)

            results.append({
                "Model": name,
                "SMOTE": use_smote,
                "Threshold": t,
                "CV_ROC_AUC": cv_score,
                "Test_ROC_AUC": roc,
                "F1_Leave": f1_score(y_test, y_pred, pos_label=1),
                "Recall_Leave": recall_score(y_test, y_pred, pos_label=1),
                "Precision_Leave": precision_score(y_test, y_pred, pos_label=1),
                "Recall_Stay": recall_score(y_test, y_pred, pos_label=0),
                "Precision_Stay": precision_score(y_test, y_pred, pos_label=0),
                
            })

# --- Results ---
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Recall_Leave", "Precision_Leave"], ascending=False)

results_df.to_csv("model_threshold_comparison.csv", index=False)

results_df

/Users/andrewzaki/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/andrewzaki/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/andrewzaki/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/andrewzaki/Library

,Model,SMOTE,Threshold,CV_ROC_AUC,Test_ROC_AUC,F1_Leave,Recall_Leave,Precision_Leave,Recall_Stay,Precision_Stay
10,RandomForest,True,0.30,0.775014,0.770867,0.408377,0.829787,0.270833,0.574899,0.946667
0,Logistic,True,0.30,0.817192,0.789129,0.420455,0.787234,0.286822,0.627530,0.939394
2,Logistic,True,0.40,0.817192,0.789129,0.483221,0.765957,0.352941,0.732794,0.942708
1,Logistic,True,0.35,0.817192,0.789129,0.458599,0.765957,0.327273,0.700405,0.940217
3,Logistic,True,0.45,0.817192,0.789129,0.500000,0.723404,0.382022,0.777328,0.936585
11,RandomForest,True,0.35,0.775014,0.770867,0.456376,0.723404,0.333333,0.724696,0.932292
20,CatBoost,True,0.30,0.781285,0.757516,0.428571,0.702128,0.308411,0.700405,0.925134
4,Logistic,True,0.50,0.817192,0.789129,0.496124,0.680851,0.390244,0.797571,0.929245
21,CatBoost,True,0.35,0.781285,0.757516,0.441379,0.680851,0.326531,0.732794,0.923469
12,RandomForest,True,0.40,0.775014,0.770867,0.496000,0.659574,0.397436,0.809717,0.925926


## 5. Final Model and Feature Importance

Train the final selected model on the full dataset to maximize performance, and analyze feature importance to understand key drivers of attrition.

- Use Logistic Regression with SMOTE and scaling for the final model  
- Retrain on the full dataset after validation  
- Extract model coefficients to identify the most influential features  
- Analyze both positive and negative drivers:
  - Positive → factors increasing attrition risk  
  - Negative → factors associated with retention  

This step provides interpretability, linking model predictions to actionable business insights.

In [19]:

final_model = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=5000))
])

final_model.fit(X, y)


joblib.dump(final_model, "attrition_model.pkl")

/Users/andrewzaki/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


['attrition_model.pkl']

In [25]:
coeffs = final_model.named_steps['model'].coef_[0]

importance = pd.Series(coeffs, index=X_train.columns)
importance = importance.sort_values(key=abs, ascending=False)

importance.head(15)

YearsAtCompany                       1.202029
OverTime                             1.032967
Department_Research & Development    0.935481
JobRole_Sales Executive              0.791198
BusinessTravel_Travel_Frequently     0.779419
BusinessTravel_Travel_Rarely         0.751141
MaritalStatus_Single                 0.727691
JobRole_Sales Representative         0.665893
TotalWorkingYears                   -0.593363
JobRole_Laboratory Technician        0.579535
JobRole_Human Resources              0.576876
YearsWithCurrManager                -0.565600
EnvironmentSatisfaction             -0.490878
JobSatisfaction                     -0.477779
Department_Sales                     0.474676
dtype: float64

In [26]:
top_positive = importance.sort_values(ascending=False).head(10)
top_negative = importance.sort_values().head(10)

top_positive, top_negative

(YearsAtCompany                       1.202029
 OverTime                             1.032967
 Department_Research & Development    0.935481
 JobRole_Sales Executive              0.791198
 BusinessTravel_Travel_Frequently     0.779419
 BusinessTravel_Travel_Rarely         0.751141
 MaritalStatus_Single                 0.727691
 JobRole_Sales Representative         0.665893
 JobRole_Laboratory Technician        0.579535
 JobRole_Human Resources              0.576876
 dtype: float64,
 TotalWorkingYears              -0.593363
 YearsWithCurrManager           -0.565600
 EnvironmentSatisfaction        -0.490878
 JobSatisfaction                -0.477779
 YearsInCurrentRole             -0.440878
 EducationField_Life Sciences   -0.440414
 JobRole_Research Director      -0.407155
 JobInvolvement                 -0.389574
 EducationField_Medical         -0.376131
 Years_per_Promotion            -0.373479
 dtype: float64)

## 6: analyze results and plot graphs

In [ ]:
scores = []
AttritionTypes =[]

for i in range(len(X)):
    res = attrition_system(final_model, X.iloc[[i]], df.iloc[i])
    scores.append(res['score'])
    AttritionTypes.append(res['AttritionType'])

pd.Series(AttritionTypes).value_counts()

No Risk                  874
Valuable Employee        245
Average Employee         146
High Value Employee ⭐    121
Low Value Employee        84
Name: count, dtype: int64